In [ ]:
# ============================================================
# 04_world_cup_2026_simulation.ipynb
# ============================================================
# World Cup 2026 Monte Carlo simulation using Gradient Boosting + FIFA ranking + historical features
# ============================================================

import pandas as pd
import numpy as np
import random
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
N_SIMULATIONS = 5000
DRAW_THRESHOLD = 0.30

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
sns.set(style="whitegrid")

ROUND_VARIANCE = {
    "group": 18,
    "play-in": 30,
    "round_of_16": 45,
    "quarterfinal": 55,
    "semifinal": 70,
    "final": 85
}

# =========================
# LOAD DATA
# =========================
model = joblib.load("../models/gradient_boosting_v1.pkl")
matches = pd.read_csv("../data/processed/matches_2000_onwards_features_fifa.csv", parse_dates=["date"])
shootouts = pd.read_csv("../data/raw/shootouts.csv", parse_dates=["date"])
fifa = pd.read_csv("../data/processed/fifa_latest_world_ranking.csv", parse_dates=["date"])
achievements = pd.read_csv("../data/raw/team_achievements.csv").set_index("team")

CURRENT_YEAR = 2026

# =========================
# UTILITY FUNCTIONS
# =========================
def achievement_score(team):
    if team not in achievements.index:
        return 0.0

    row = achievements.loc[team]
    score = 0.0

    # =========================
    # WORLD CUP (máximo ~2.5)
    # =========================
    if not np.isnan(row["wc_last_semi_year"]):
        wc_semi_recency = CURRENT_YEAR - row["wc_last_semi_year"]
        score += 0.6 * np.exp(-wc_semi_recency / 6)

    if not np.isnan(row["wc_last_final_year"]):
        wc_final_recency = CURRENT_YEAR - row["wc_last_final_year"]
        score += 0.9 * np.exp(-wc_final_recency / 6)

    if not np.isnan(row["wc_last_win_year"]):
        wc_win_recency = CURRENT_YEAR - row["wc_last_win_year"]
        score += 1.0 * np.exp(-wc_win_recency / 8)

    # =========================
    # CONTINENTAL (máximo ~1.8)
    # =========================
    if not np.isnan(row["cont_last_semi_year"]):
        cont_semi_recency = CURRENT_YEAR - row["cont_last_semi_year"]
        score += 0.4 * np.exp(-cont_semi_recency / 5)

    if not np.isnan(row["cont_last_final_year"]):
        cont_final_recency = CURRENT_YEAR - row["cont_last_final_year"]
        score += 0.6 * np.exp(-cont_final_recency / 5)

    if not np.isnan(row["cont_last_win_year"]):
        cont_win_recency = CURRENT_YEAR - row["cont_last_win_year"]
        score += 0.8 * np.exp(-cont_win_recency / 6)

    return score


def historical_penalty(team):
    if team not in achievements.index:
        return 0.0

    row = achievements.loc[team]

    # Si nunca llegó ni a semis de nada y lleva >20 años sin WC
    if (
        np.isnan(row["wc_last_semi_year"]) and
        np.isnan(row["cont_last_semi_year"]) and
        (CURRENT_YEAR - row["last_world_cup_participation"] > 20)
    ):
        return -5

    return 0.0


def smooth_form(win_rate, alpha=0.65):
    return alpha*win_rate + (1-alpha)*0.5

def modern_strength(team):
    score = 0.0

    # =========================
    # 1. FORMA RECIENTE (base)
    # =========================
    score += (smooth_form(last5_win_rate.get(team, 0.5)) - 0.5) * 110

    if team not in achievements.index:
        return score

    row = achievements.loc[team]

    # =========================
    # 2. WORLD CUP MODERNA
    # =========================
    if not np.isnan(row["wc_last_semi_year"]):
        years = CURRENT_YEAR - row["wc_last_semi_year"]
        score += 22 * np.exp(-years / 6)

    if not np.isnan(row["wc_last_final_year"]):
        years = CURRENT_YEAR - row["wc_last_final_year"]
        score += 28 * np.exp(-years / 6)

    if not np.isnan(row["wc_last_win_year"]):
        years = CURRENT_YEAR - row["wc_last_win_year"]
        score += 32 * np.exp(-years / 8)

    # =========================
    # 3. CONTINENTAL MODERNA
    # =========================
    if not np.isnan(row["cont_last_semi_year"]):
        years = CURRENT_YEAR - row["cont_last_semi_year"]
        score += 14 * np.exp(-years / 5)

    if not np.isnan(row["cont_last_final_year"]):
        years = CURRENT_YEAR - row["cont_last_final_year"]
        score += 18 * np.exp(-years / 5)

    if not np.isnan(row["cont_last_win_year"]):
        years = CURRENT_YEAR - row["cont_last_win_year"]
        score += 22 * np.exp(-years / 6)

    # =========================
    # 4. MICRO AJUSTE MANUAL (opcional)
    # =========================
    MODERN_TEAMS = {
        "Argentina": 6,
        "Spain": 6,
        "France": 6,
        "England": 5,
        "Morocco": 5,
        "Portugal": 5,
        "Germany": 4,
        "Brazil": 4,
        "Belgium": 4,
        "Netherlands": 4,
        "Croatia": 4,
        "Japan": 3,
        "Senegal": 3,
        "Uruguay": 3
    }

    score += MODERN_TEAMS.get(team, 0)

    return score


# FIFA points
fifa_points = dict(zip(fifa["country"], fifa["total_points"]))

# Penalty win rate
penalty_wins = shootouts["winner"].value_counts()
penalty_games = pd.concat([shootouts["home_team"], shootouts["away_team"]]).value_counts()
penalty_win_rate = (penalty_wins / penalty_games).fillna(0.5)

# Last5 form
last5_win_rate = {}
for team, group in matches.groupby("home_team"):
    last5_win_rate[team] = group["home_last5_win_rate"].iloc[-1] if len(group) else 0.5
for team, group in matches.groupby("away_team"):
    last5_win_rate[team] = group["away_last5_win_rate"].iloc[-1] if len(group) else 0.5

# H2H
h2h_stats = {}
for idx, row in matches.iterrows():
    key = tuple(sorted([row["home_team"], row["away_team"]]))
    h2h_stats[key] = {"home_win_rate": row["h2h_home_win_rate"], "draw_rate": row["h2h_draw_rate"], "matches_played": max(1,row["h2h_matches_played"])}

# =========================
# PLAYOFF TEAMS
# =========================
playoff_candidates = {
    "UEFA_Playoff_A":["Italy","Bosnia and Herzegovina","Northern Ireland","Wales"],
    "UEFA_Playoff_B":["Albania","Poland","Sweden","Ukraine"],
    "UEFA_Playoff_C":["Kosovo","Romania","Slovakia","Turkey"],
    "UEFA_Playoff_D":["Denmark","Republic of Ireland","North Macedonia","Czech Republic"],
    "FIFA_Playoff_1":["Jamaica","New Caledonia","DR Congo"],
    "FIFA_Playoff_2":["Bolivia","Iraq","Suriname"]
}
for playoff, teams in playoff_candidates.items():
    fifa_points[playoff] = np.mean([fifa_points[t] for t in teams])
    last5_win_rate[playoff] = np.mean([last5_win_rate.get(t,0.5) for t in teams])

# =========================
# GROUPS
# =========================
groups = {
    "A":["Mexico","South Africa","South Korea","UEFA_Playoff_D"],
    "B":["Canada","Switzerland","Qatar","UEFA_Playoff_A"],
    "C":["Brazil","Morocco","Haiti","Scotland"],
    "D":["United States","Paraguay","Australia","UEFA_Playoff_C"],
    "E":["Germany","Ivory Coast","Ecuador","Curaçao"],
    "F":["Netherlands","Japan","Tunisia","UEFA_Playoff_B"],
    "G":["Belgium","Egypt","Iran","New Zealand"],
    "H":["Spain","Uruguay","Saudi Arabia","Cape Verde"],
    "I":["France","Senegal","Norway","FIFA_Playoff_2"],
    "J":["Argentina","Algeria","Austria","Jordan"],
    "K":["Portugal","Colombia","Uzbekistan","FIFA_Playoff_1"],
    "L":["England","Croatia","Ghana","Panama"]
}

# =========================
# SIMULATION FUNCTIONS
# =========================
def simulate_match(team_a, team_b, model, round_name="group", draw_threshold=DRAW_THRESHOLD):
    key = tuple(sorted([team_a, team_b]))
    h2h = h2h_stats.get(key, {"home_win_rate":0.5,"draw_rate":0.0,"matches_played":1})
    fifa_diff = fifa_points.get(team_a,1500) - fifa_points.get(team_b,1500)
    form_a = smooth_form(last5_win_rate.get(team_a,0.5))
    form_b = smooth_form(last5_win_rate.get(team_b,0.5))
    fifa_diff += np.random.normal(0, ROUND_VARIANCE[round_name])
    fifa_diff += historical_penalty(team_a) - historical_penalty(team_b)
    h2h_weight = min(1.0,h2h["matches_played"]/10)
    h2h_effective = h2h["home_win_rate"]*h2h_weight

    features = pd.DataFrame([{
        "fifa_diff": fifa_diff,
        "home_last5_win_rate": form_a,
        "away_last5_win_rate": form_b,
        "h2h_home_win_rate": h2h["home_win_rate"],
        "h2h_draw_rate": h2h["draw_rate"],
        "h2h_matches_played": h2h["matches_played"],
        "home_penalty_win_rate": penalty_win_rate.get(team_a,0.5),
        "away_penalty_win_rate": penalty_win_rate.get(team_b,0.5),
        "neutral": True,
        "fifa_diff_x_home_form": fifa_diff*form_a,
        "fifa_diff_x_away_form": fifa_diff*form_b,
        "h2h_effective": h2h_effective
    }])

    p_away, p_draw, p_home = model.predict_proba(features)[0]

    # Modern football adjustment
    modern_diff = modern_strength(team_a) - modern_strength(team_b)
    modern_boost = 1 / (1 + np.exp(-modern_diff / 70))
    p_home *= modern_boost
    p_away *= (1 - modern_boost)

    # Renormalizar
    total = p_home + p_away
    p_home /= total
    p_away /= total

    p_home = np.clip(p_home, 0.15, 0.85)
    p_away = np.clip(p_away, 0.15, 0.85)

    if p_draw > draw_threshold:
        return "D"
    return "A" if np.random.rand() < p_home / (p_home + p_away) else "B"

def simulate_group(group_teams):
    table = {team: {"points":0,"gd":0} for team in group_teams}
    for i in range(len(group_teams)):
        for j in range(i+1,len(group_teams)):
            t1,t2=group_teams[i],group_teams[j]
            result=simulate_match(t1,t2,model)
            if result=="A": table[t1]["points"]+=3; table[t1]["gd"]+=1; table[t2]["gd"]-=1
            elif result=="B": table[t2]["points"]+=3; table[t2]["gd"]+=1; table[t1]["gd"]-=1
            else: table[t1]["points"]+=1; table[t2]["points"]+=1
    return pd.DataFrame(table).T.sort_values(["points","gd"],ascending=False)

def simulate_knockout(team_a,team_b,round_name):
    result=simulate_match(team_a,team_b,model,round_name)
    if result=="D":
        pa=penalty_win_rate.get(team_a,0.5)
        pb=penalty_win_rate.get(team_b,0.5)
        return team_a if np.random.rand()<pa/(pa+pb) else team_b
    return team_a if result=="A" else team_b

def simulate_tournament():
    group_tables = [simulate_group(g) for g in groups.values()]
    firsts = [g.index[0] for g in group_tables]

    seconds = []
    for idx, g in enumerate(group_tables):
        seconds.append({
            "team": g.index[1],
            "points": g.iloc[1]["points"],
            "gd": g.iloc[1]["gd"],
            "group": idx
        })
    seconds_df = pd.DataFrame(seconds)
    seconds_df["tie_break"] = np.random.rand(len(seconds_df))
    seconds_df = seconds_df.sort_values(["points", "gd", "tie_break"], ascending=[False, False, True])

    best_seconds = seconds_df.head(8)["team"].tolist()
    playin_seconds = seconds_df.tail(len(seconds_df)-8)["team"].tolist()

    random.shuffle(playin_seconds)
    playin_winners = []
    i = 0
    while i < len(playin_seconds):
        if i+1 < len(playin_seconds):
            winner = simulate_knockout(playin_seconds[i], playin_seconds[i+1], "play-in")
            playin_winners.append(winner)
            i += 2
        else:
            playin_winners.append(playin_seconds[i])
            i += 1

    qualified = firsts + best_seconds + playin_winners
    random.shuffle(qualified)
    knockout_rounds = [("round_of_16", 16), ("quarterfinal", 8), ("semifinal", 4), ("final", 2)]

    for round_name, _ in knockout_rounds:
        next_round = []
        i = 0
        while i < len(qualified):
            if i+1 < len(qualified):
                next_round.append(simulate_knockout(qualified[i], qualified[i+1], round_name))
                i += 2
            else:
                next_round.append(qualified[i])
                i += 1
        qualified = next_round

    return qualified[0]

# =========================
# MONTE CARLO
# =========================
winners=[simulate_tournament() for _ in range(N_SIMULATIONS)]
winner_counts = pd.Series(winners).value_counts(normalize=True)*100
top_winners = winner_counts.head(10)

plt.figure(figsize=(12,6))
sns.barplot(x=top_winners.values,y=top_winners.index,palette="viridis")
plt.title("World Cup 2026 — Champion Probability (%)")
plt.xlabel("Probability (%)")
plt.ylabel("Team")
plt.tight_layout()
plt.show()

winner_counts.head(15)

results_file = "../simulation_results/world_cup_2026_winner_probabilities.csv"

winner_counts_df = winner_counts.reset_index()
winner_counts_df.columns = ["team", "probability_percent"]
winner_counts_df = winner_counts_df.sort_values("probability_percent", ascending=False)

winner_counts_df.to_csv(results_file, index=False)

print(f"Results saved to {results_file}")